### **Notebook 7 (etapa 4): Entrenamiento de modelos estrictos de severidad y consumo de recursos**

In [ ]:
import pandas as pd  # Permite el manejo y análisis de estructuras de datos (DataFrames)
import numpy as np  # Facilita la realización de cálculos numéricos y el manejo de matrices
import os  # Interacción con el sistema operativo (creación y verificación de rutas/directorios)
import xgboost as xgb  # Algoritmo de ensamble avanzado (Gradient Boosting) para entrenamiento predictivo
from sklearn.model_selection import GroupShuffleSplit  # Herramienta para partición de datos (importada por consistencia del entorno)
from sklearn.preprocessing import label_binarize  # Convierte etiquetas multiclase a formato binario (One-vs-Rest)
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score  # Métricas para evaluar el rendimiento

# 1. Configuración de rutas
# Directorio donde se ubican los datasets de entrenamiento y prueba ya particionados
dir_datos = "../../Datos/Datasets Finales"

print("=== ESTUDIO DE ABLACIÓN: CORRECCIÓN DE CIRCULARIDAD DEFINICIONAL ===")

# Variables a EXCLUIR explícitamente para evitar la "trampa del GRD" (Data Leakage definicional)
# Estas variables son los insumos matemáticos que el agrupador usa para calcular Severidad y Consumo
vars_circulares = ['NUM_COMORBILIDADES', 'NUM_PROCEDIMIENTOS', 'DIAS_ESTADIA']
# Nota: COMORBILIDAD_PRINCIPAL ya está en formato One-Hot, así que buscaremos todas sus columnas derivadas.

# 2. Cargar datos oncológicos de Entrenamiento y Prueba
print("Cargando datasets oncológicos...")
# Cargar el set de entrenamiento oncológico 
df_onco_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_onco.csv"), low_memory=False)
# Cargar el set de prueba oncológico
df_onco_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), low_memory=False)

# Identificar dinámicamente todas las columnas OHE derivadas de la variable COMORBILIDAD_PRINCIPAL
cols_comorbilidad = [col for col in df_onco_train.columns if col.startswith('COMORBILIDAD_PRINCIPAL_')]
# Consolidar la lista maestra de variables que no deben ser vistas por el modelo
todas_vars_prohibidas = vars_circulares + cols_comorbilidad

# Informar la magnitud de la ablación (cuántos atributos se eliminarán)
print(f"Se eliminarán {len(todas_vars_prohibidas)} variables asociadas al cálculo interno del GRD.")

def entrenar_evaluar_estricto(target_name, df_train, df_test, prohibidas):
    """
    Descripción:
        Entrena y evalúa un modelo clínico "estricto" (XGBoost) para predecir variables del GRD 
        (Severidad o Consumo). Este enfoque elimina de la matriz predictora todas aquellas variables 
        que generan circularidad definicional, asegurando que el modelo aprenda patrones clínicos 
        reales en lugar de memorizar la fórmula interna del algoritmo GRD.

    Entradas:
        - target_name (str): Nombre de la variable objetivo multiclase a predecir.
        - df_train (DataFrame): Conjunto de datos de entrenamiento oncológico.
        - df_test (DataFrame): Conjunto de datos de prueba oncológico (cohorte de validación).
        - prohibidas (list): Lista de nombres de columnas que deben excluirse por circularidad.

    Salidas:
        - Tupla (f1_macro, auc, auprc): Valores flotantes correspondientes a las métricas de 
          desempeño del modelo estricto, devueltas tras su validación.
    """
    print(f"\n--- Entrenando Modelo Clínico Estricto: {target_name} ---")
    
    # Preparar listas de columnas a dropear: Metadatos, todos los posibles targets y la lista de prohibidas
    cols_drop = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CIP_ENCRIPTADO', 'CATEGORIA_CANCER'] + prohibidas
    
    # Construir la matriz de características (X) y el vector objetivo (y) de Entrenamiento
    X_train = df_train.drop(columns=cols_drop, errors='ignore')
    y_train = df_train[target_name]
    
    # Construir la matriz de características (X) y el vector objetivo (y) de Prueba
    X_test = df_test.drop(columns=cols_drop, errors='ignore')
    # Alinear las columnas de Prueba con Entrenamiento rellenando con 0 si faltara alguna (seguridad para OHE)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    y_test = df_test[target_name]
    
    # Instanciar el modelo XGBoost utilizando los hiperparámetros óptimos previamente consolidados
    modelo = xgb.XGBClassifier(
        learning_rate=0.3,      # Tasa de aprendizaje óptima encontrada
        max_depth=10,           # Profundidad máxima estable
        tree_method='hist',     # Método de histogramas para agilizar el procesamiento
        n_jobs=-1,              # Utilización máxima de procesadores
        random_state=42         # Semilla de reproducibilidad
    )
    # Ejecutar el ajuste matemático (entrenamiento) sobre la matriz "limpia" de sesgos
    modelo.fit(X_train, y_train)
    
    # Inferir clases duras (predicciones) sobre el set de prueba
    y_pred = modelo.predict(X_test)
    # Inferir matriz de probabilidades continuas para todas las clases
    y_prob = modelo.predict_proba(X_test)
    
    # Extraer las clases únicas reales presentes en el test
    clases = np.unique(y_test)
    # Binarizar el vector de prueba (One-vs-Rest) para el cálculo de AUPRC multiclase
    y_test_bin = label_binarize(y_test, classes=clases)
    
    # Calcular métricas globales (F1-Macro, AUC ponderado y AUPRC ponderado)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')
    auprc = average_precision_score(y_test_bin, y_prob, average='weighted')
    
    # Mostrar resultados del modelo libre de variables circulares
    print(f"F1-Macro Estricto: {f1_macro:.4f}")
    print(f"AUC-ROC Estricto:  {auc:.4f}")
    print(f"AUPRC Estricto:    {auprc:.4f}")
    
    # Retornar las métricas por si requirieran ser almacenadas o comparadas en el flujo principal
    return f1_macro, auc, auprc

# ====================================================================
# 3. Ejecutar Evaluación Estricta para Severidad
# ====================================================================
# Llama a la función excluyendo las variables circulares para predecir Severidad
f1_sev, auc_sev, auprc_sev = entrenar_evaluar_estricto('SEVERIDAD', df_onco_train, df_onco_test, todas_vars_prohibidas)

# ====================================================================
# 4. Ejecutar Evaluación Estricta para Consumo de Recursos
# ====================================================================
# Llama a la función excluyendo las variables circulares para predecir Consumo de Recursos
f1_cons, auc_cons, auprc_cons = entrenar_evaluar_estricto('CONSUMO_RECURSOS', df_onco_train, df_onco_test, todas_vars_prohibidas)

=== ESTUDIO DE ABLACIÓN: CORRECCIÓN DE CIRCULARIDAD DEFINICIONAL ===
Cargando datasets oncológicos...
Se eliminarán 20 variables asociadas al cálculo interno del GRD.

--- Entrenando Modelo Clínico Estricto: SEVERIDAD ---
F1-Macro Estricto: 0.6716
AUC-ROC Estricto:  0.8538
AUPRC Estricto:    0.7161

--- Entrenando Modelo Clínico Estricto: CONSUMO_RECURSOS ---
F1-Macro Estricto: 0.6799
AUC-ROC Estricto:  0.8515
AUPRC Estricto:    0.8158


In [ ]:
import pandas as pd  # Permite el manejo y análisis de estructuras de datos (DataFrames)
import numpy as np  # Facilita la realización de cálculos numéricos y el manejo de matrices
import os  # Interacción con el sistema operativo (creación y verificación de rutas/directorios)
import joblib  # Serialización y guardado de los modelos entrenados de Machine Learning en disco
import xgboost as xgb  # Algoritmo de ensamble avanzado (Gradient Boosting) para entrenamiento predictivo
from sklearn.preprocessing import label_binarize  # Convierte etiquetas multiclase a formato binario (One-vs-Rest)
from sklearn.metrics import f1_score, roc_auc_score  # Métricas para evaluar el rendimiento rápido del modelo

# 1. Configuración de rutas
# Directorio donde se ubican los datasets preprocesados y particionados
dir_datos = "../../Datos/Datasets Finales"
# Directorio destino donde se guardarán los modelos estrictos (Ablación)
dir_modelos = "../../Resultados/Resultados (etapa 3 y 4)/XGBoost"
# Crear el directorio de modelos si es que aún no existe en el sistema
os.makedirs(dir_modelos, exist_ok=True)

print("=== ESTUDIO DE ABLACIÓN: ENTRENANDO Y GUARDANDO MODELOS ESTRICTOS ===")

# Variables a EXCLUIR para eliminar la circularidad definicional
# (Insumos directos de la fórmula matemática del agrupador GRD)
vars_circulares = ['NUM_COMORBILIDADES', 'NUM_PROCEDIMIENTOS', 'DIAS_ESTADIA']

# 2. Cargar datos oncológicos
print("Cargando datasets oncológicos...")
# Cargar la cohorte de entrenamiento exclusiva para pacientes oncológicos
df_onco_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_onco.csv"), low_memory=False)
# Cargar la cohorte de prueba exclusiva para pacientes oncológicos
df_onco_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), low_memory=False)

# Identificar dinámicamente todas las columnas One-Hot Encoding que derivan de la comorbilidad principal
cols_comorbilidad = [col for col in df_onco_train.columns if col.startswith('COMORBILIDAD_PRINCIPAL_')]
# Consolidar la lista maestra de variables "prohibidas" que no deben ingresar al modelo
todas_vars_prohibidas = vars_circulares + cols_comorbilidad

def entrenar_guardar_estricto(target_name, df_train, df_test, prohibidas):
    """
    Descripción:
        Entrena un modelo predictivo clínico "estricto" (XGBoost) excluyendo aquellas variables 
        que generan circularidad. Una vez ajustado el modelo, lo serializa y guarda físicamente 
        en disco (.pkl) para que pueda ser utilizado posteriormente (ej. análisis de explicabilidad 
        con SHAP). Finalmente, ejecuta una evaluación rápida para confirmar su integridad.

    Entradas:
        - target_name (str): Nombre de la variable objetivo multiclase a predecir.
        - df_train (DataFrame): Conjunto de datos de entrenamiento oncológico.
        - df_test (DataFrame): Conjunto de datos de prueba oncológico (validación).
        - prohibidas (list): Lista de nombres de columnas a excluir por sesgo de circularidad.

    Salidas:
        - None: La función no retorna variables en memoria, pero genera un archivo .pkl 
          en el directorio local e imprime por consola un resumen del rendimiento.
    """
    print(f"\n--- Entrenando Modelo Clínico Estricto: {target_name} ---")
    
    # Consolidar la lista de columnas a eliminar: otros targets, metadatos y las variables sesgadas
    cols_drop = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CIP_ENCRIPTADO', 'CATEGORIA_CANCER'] + prohibidas
    
    # Preparar matriz de características (X) y vector objetivo (y) para Entrenamiento
    X_train = df_train.drop(columns=cols_drop, errors='ignore')
    y_train = df_train[target_name]
    
    # Preparar matriz de características (X) y vector objetivo (y) para Prueba
    X_test = df_test.drop(columns=cols_drop, errors='ignore')
    # Alinear estrictamente las columnas de Prueba con las de Entrenamiento (rellenando vacíos con 0)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    y_test = df_test[target_name]
    
    # Entrenar: Instanciar XGBoost con los hiperparámetros óptimos identificados en etapas previas
    modelo = xgb.XGBClassifier(
        learning_rate=0.3,      # Tasa de aprendizaje estable y ganadora
        max_depth=10,           # Profundidad máxima permitida por árbol
        tree_method='hist',     # Uso de histogramas para mayor eficiencia de cómputo
        n_jobs=-1,              # Explota todos los núcleos del servidor
        random_state=42         # Semilla fija para reproducibilidad de resultados
    )
    # Ejecutar el ajuste matemático con la matriz limpia
    modelo.fit(X_train, y_train)
    
    # --- GUARDAR EL MODELO ESTRICTO EN DISCO ---
    # Construir la ruta dinámica con el sufijo "ESTRICTO" para no sobrescribir el modelo original
    ruta_modelo = os.path.join(dir_modelos, f"Modelo_Optimo_XGBoost_{target_name}_ESTRICTO.pkl")
    # Exportar (serializar) el objeto del modelo usando joblib
    joblib.dump(modelo, ruta_modelo)
    print(f"-> Modelo guardado exitosamente en: {ruta_modelo}")
    
    # Evaluar rápido para confirmar integridad del modelo recién guardado
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)
    
    # Extraer clases únicas y binarizar el target para cálculo de AUC multiclase
    clases = np.unique(y_test)
    y_test_bin = label_binarize(y_test, classes=clases)
    
    # Calcular métricas globales de evaluación rápida
    f1_macro = f1_score(y_test, y_pred, average='macro')
    auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')
    
    # Imprimir el rendimiento estricto
    print(f"F1-Macro Estricto: {f1_macro:.4f} | AUC-ROC: {auc:.4f}")

# ====================================================================
# 3. Ejecutar entrenamiento, guardado y validación por cada Target
# ====================================================================

# Ejecutar el flujo completo para predecir la Severidad libre de sesgos
entrenar_guardar_estricto('SEVERIDAD', df_onco_train, df_onco_test, todas_vars_prohibidas)

# Ejecutar el flujo completo para predecir el Consumo de Recursos libre de sesgos
entrenar_guardar_estricto('CONSUMO_RECURSOS', df_onco_train, df_onco_test, todas_vars_prohibidas)

# Mensaje finalizando el proceso de ablación y serialización
print("\n¡Modelos estrictos generados y guardados!")

=== ESTUDIO DE ABLACIÓN: ENTRENANDO Y GUARDANDO MODELOS ESTRICTOS ===
Cargando datasets oncológicos...

--- Entrenando Modelo Clínico Estricto: SEVERIDAD ---
-> Modelo guardado exitosamente en: ../../Resultados/Resultados (etapa 3 y 4)/XGBoost/Modelo_Optimo_XGBoost_SEVERIDAD_ESTRICTO.pkl
F1-Macro Estricto: 0.6716 | AUC-ROC: 0.8538

--- Entrenando Modelo Clínico Estricto: CONSUMO_RECURSOS ---
-> Modelo guardado exitosamente en: ../../Resultados/Resultados (etapa 3 y 4)/XGBoost/Modelo_Optimo_XGBoost_CONSUMO_RECURSOS_ESTRICTO.pkl
F1-Macro Estricto: 0.6799 | AUC-ROC: 0.8515

¡Modelos estrictos generados y guardados!
